# Huấn Luyện UC-TTT & Ablation Studies Trên Google Colab
## (User-Credibility Temporal Tree Transformer — Nhóm 6)

**Mục đích notebook này**: Chạy **UC-TTT Proposed** và **9 biến thể Ablation** trên Colab T4.

> ⚠️ TTT Baseline và các Baselines (TF-IDF, RF, XGBoost) **đang chạy trên máy local** — KHÔNG cần chạy ở đây.

### Các bước:
1. Kiểm tra GPU → Mount Drive → Clone repo → Cài thư viện
2. Nạp dữ liệu từ Google Drive
3. Chạy **UC-TTT** (residual_gate, 5-fold LOEO, 5,447 threads)
4. Chạy **từng biến thể Ablation** riêng lẻ (hoặc tất cả cùng lúc)
5. Sinh biểu đồ → Sao lưu Drive → Download JSON về máy

---

## 🔧 SETUP

### Bước 1: Kiểm tra GPU

In [ ]:
!nvidia-smi

### Bước 2: Kết nối Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Bước 3: Clone Repo từ GitHub

In [ ]:
%cd /content
!rm -rf repo
!git clone -b feat/unified-mini-proposal-and-error-analysis \
    https://github.com/tphu2029/SNA_EnhancingTTT-via-User-Credibility-Integration-for-Social-Media-Rumor-Detection.git repo
%cd /content/repo
!git log --oneline -3

### Bước 4: Cài đặt thư viện

In [ ]:
!pip install -q "transformers>=4.30.0,<4.49.0" scikit-learn pandas networkx tqdm

### Bước 5: Nạp dữ liệu từ Google Drive

> ⚠️ Nếu thư mục `PHEME_DATA` nằm trong **"Được chia sẻ với tôi"**, hãy tạo **Lối tắt (Shortcut)** vào **"Drive của tôi"** trước:
> Drive → Được chia sẻ với tôi → Chuột phải vào `PHEME_DATA` → Tổ chức → Thêm lối tắt → My Drive

In [ ]:
import os, shutil

CANDIDATE_DIRS = [
    '/content/drive/MyDrive/MXH/PHEME_DATA',
    '/content/drive/MyDrive/PHEME_DATA',
    '/content/drive/MyDrive/Colab Notebooks/PHEME_DATA',
    '/content/drive/MyDrive/PHEME',
    '/content/drive/MyDrive/MXH',
]

DRIVE_DATA_DIR = None
for d in CANDIDATE_DIRS:
    if os.path.exists(d) and (
        os.path.exists(os.path.join(d, 'pheme_processed.pkl')) or
        os.path.exists(os.path.join(d, 'text_embeddings_bert.pt'))
    ):
        DRIVE_DATA_DIR = d
        print(f"✅ Tìm thấy dữ liệu tại: {DRIVE_DATA_DIR}")
        break

if DRIVE_DATA_DIR is None:
    print("Không thấy ở đường dẫn chuẩn, đang quét toàn Drive...")
    for root, dirs, files in os.walk('/content/drive'):
        if 'pheme_processed.pkl' in files or 'text_embeddings_bert.pt' in files:
            DRIVE_DATA_DIR = root
            print(f"✅ [Tự động] Tìm thấy: {DRIVE_DATA_DIR}")
            break

if DRIVE_DATA_DIR is None:
    raise FileNotFoundError("❌ Không tìm thấy dữ liệu PHEME trên Drive! Hãy tạo Lối tắt vào My Drive theo hướng dẫn ở trên.")

os.makedirs('data/processed', exist_ok=True)

for fname, dst in [
    ('pheme_processed.pkl',     'data/processed/pheme_processed.pkl'),
    ('text_embeddings_bert.pt', 'data/processed/text_embeddings_bert.pt'),
]:
    src = os.path.join(DRIVE_DATA_DIR, fname)
    if os.path.exists(src):
        shutil.copyfile(src, dst)
        print(f"✅ {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)")
    else:
        print(f"⚠️  {fname} — không tìm thấy trong {DRIVE_DATA_DIR}")

print("\n🎉 Nạp dữ liệu hoàn tất! Tiến hành Bước 5.2 nếu thiếu text_embeddings_bert.pt, ngược lại nhảy tới Bước 6.")

### Bước 5.2 (TÙY CHỌN): Trích xuất BERT Embedding
> Chỉ cần chạy khi **Bước 5 báo thiếu `text_embeddings_bert.pt`**.

In [ ]:
if os.path.exists('data/processed/text_embeddings_bert.pt'):
    print("✅ text_embeddings_bert.pt đã có — bỏ qua bước này.")
else:
    print("⏳ Trích xuất BERT embeddings (~2 phút)...")
    !python experiments/extract_text_embeddings.py \
        --model_name bert-base-uncased \
        --batch_size 128 \
        --pooling max \
        --output data/processed/text_embeddings_bert.pt
    # Sao lưu lên Drive
    if os.path.exists('data/processed/text_embeddings_bert.pt') and DRIVE_DATA_DIR:
        shutil.copyfile('data/processed/text_embeddings_bert.pt',
                        os.path.join(DRIVE_DATA_DIR, 'text_embeddings_bert.pt'))
        print("✅ Đã sao lưu text_embeddings_bert.pt lên Drive.")

---
## 🚀 THỰC NGHIỆM

> **Lưu ý chung**: Tất cả lệnh dùng `--min_reactions 1` → lọc đúng **5,447 threads** khớp Wu et al. (2025).

---
## Bước 6: UC-TTT Proposed

**UC-TTT** với Residual-Gate Fusion, 5-fold LOEO, 5,447 threads.  
ETA: ~30 phút/fold × 5 = **~2.5h** trên T4.

In [ ]:
!python -u experiments/train_ucttt.py \
    --embeddings_path data/processed/text_embeddings_bert.pt \
    --events_mode 5 \
    --window_mode fixed_posts \
    --window_size 8 \
    --fusion_type residual_gate \
    --d_model 768 \
    --nhead 12 \
    --num_tree_layers 6 \
    --val_ratio 0.30 \
    --checkpoint_criterion loss_stabilize \
    --epochs 35 \
    --patience 8 \
    --min_reactions 1

---
## Bước 7: Ablation Studies

9 biến thể bóc tách — ETA ~**22–25h** nếu chạy tất cả.  
**Khuyến nghị**: Chạy từng cell riêng lẻ theo thứ tự, checkpoint được lưu tự động nên có thể tiếp tục nếu Colab ngắt kết nối.

### 7A. Feature-level Ablation — `counts_only`
Chỉ dùng 5 Engagement Counts (followers, friends, statuses, listed, favourites).  
*(Đây là biến thể tốt nhất theo kết quả nội bộ)*

In [ ]:
!python -u experiments/run_ablation.py \
    --feature_group counts_only \
    --events_mode 5 \
    --embeddings_path data/processed/text_embeddings_bert.pt \
    --fusion_type residual_gate \
    --val_ratio 0.30 \
    --checkpoint_criterion loss_stabilize \
    --epochs 35 \
    --patience 8 \
    --min_reactions 1

### 7B. Feature-level Ablation — `without_ratios`
Loại bỏ 3 Derived Ratios → còn 9 đặc trưng.

In [ ]:
!python -u experiments/run_ablation.py \
    --feature_group without_ratios \
    --events_mode 5 \
    --embeddings_path data/processed/text_embeddings_bert.pt \
    --fusion_type residual_gate \
    --val_ratio 0.30 \
    --checkpoint_criterion loss_stabilize \
    --epochs 35 \
    --patience 8 \
    --min_reactions 1

### 7C. Feature-level Ablation — `without_profile`
Loại bỏ 4 Profile & Verification features → còn 8 đặc trưng.

In [ ]:
!python -u experiments/run_ablation.py \
    --feature_group without_profile \
    --events_mode 5 \
    --embeddings_path data/processed/text_embeddings_bert.pt \
    --fusion_type residual_gate \
    --val_ratio 0.30 \
    --checkpoint_criterion loss_stabilize \
    --epochs 35 \
    --patience 8 \
    --min_reactions 1

### 7D. Feature-level Ablation — `without_counts`
Loại bỏ 5 Engagement Counts → còn 7 đặc trưng.

In [ ]:
!python -u experiments/run_ablation.py \
    --feature_group without_counts \
    --events_mode 5 \
    --embeddings_path data/processed/text_embeddings_bert.pt \
    --fusion_type residual_gate \
    --val_ratio 0.30 \
    --checkpoint_criterion loss_stabilize \
    --epochs 35 \
    --patience 8 \
    --min_reactions 1

### 7E. Feature-level Ablation — `profile_only`
Chỉ dùng 4 Profile & Verification features.

In [ ]:
!python -u experiments/run_ablation.py \
    --feature_group profile_only \
    --events_mode 5 \
    --embeddings_path data/processed/text_embeddings_bert.pt \
    --fusion_type residual_gate \
    --val_ratio 0.30 \
    --checkpoint_criterion loss_stabilize \
    --epochs 35 \
    --patience 8 \
    --min_reactions 1

### 7F. Feature-level Ablation — `ratios_only`
Chỉ dùng 3 Derived Ratios (reputation_ratio, activity_rate, media_rate).

In [ ]:
!python -u experiments/run_ablation.py \
    --feature_group ratios_only \
    --events_mode 5 \
    --embeddings_path data/processed/text_embeddings_bert.pt \
    --fusion_type residual_gate \
    --val_ratio 0.30 \
    --checkpoint_criterion loss_stabilize \
    --epochs 35 \
    --patience 8 \
    --min_reactions 1

### 7G. Feature-level Ablation — `source_only`
Chỉ dùng thông tin user của tweet gốc (root node), reactions đặt về 0.

In [ ]:
!python -u experiments/run_ablation.py \
    --feature_group source_only \
    --events_mode 5 \
    --embeddings_path data/processed/text_embeddings_bert.pt \
    --fusion_type residual_gate \
    --val_ratio 0.30 \
    --checkpoint_criterion loss_stabilize \
    --epochs 35 \
    --patience 8 \
    --min_reactions 1

### 7H. Architecture-level Ablation — `without_temporal_gru`
Bỏ Temporal GRU — chỉ dùng Tree Transformer tĩnh.

In [ ]:
!python -u experiments/run_ablation.py \
    --model_component without_temporal_gru \
    --events_mode 5 \
    --embeddings_path data/processed/text_embeddings_bert.pt \
    --fusion_type residual_gate \
    --val_ratio 0.30 \
    --checkpoint_criterion loss_stabilize \
    --epochs 35 \
    --patience 8 \
    --min_reactions 1

### 7I. Architecture-level Ablation — `without_tree_transformer`
Bỏ Tree Transformer — chỉ dùng GRU chuỗi thời gian.

In [ ]:
!python -u experiments/run_ablation.py \
    --model_component without_tree_transformer \
    --events_mode 5 \
    --embeddings_path data/processed/text_embeddings_bert.pt \
    --fusion_type residual_gate \
    --val_ratio 0.30 \
    --checkpoint_criterion loss_stabilize \
    --epochs 35 \
    --patience 8 \
    --min_reactions 1

---
### 7-ALL: Chạy TẤT CẢ 9 biến thể tự động (thay thế 7A–7I)
> ⚠️ **ETA ~22–25h** trên T4. Bật `Colab Pro` hoặc dùng session không ngắt.

In [ ]:
!python -u experiments/run_ablation.py \
    --run_all \
    --events_mode 5 \
    --embeddings_path data/processed/text_embeddings_bert.pt \
    --fusion_type residual_gate \
    --val_ratio 0.30 \
    --checkpoint_criterion loss_stabilize \
    --epochs 35 \
    --patience 8 \
    --min_reactions 1

---
## Bước 8: Sinh Biểu Đồ Ablation

In [ ]:
from IPython.display import Image, display

!python experiments/plot_ablation_results.py \
    --input results/ablation/ablation_summary.json \
    --output results/figures/ablation_feature_contribution.png

for fig in ['results/figures/ablation_feature_contribution.png',
            'results/figures/ablation_full_summary.png']:
    if os.path.exists(fig):
        display(Image(fig, width=950))

---
## Bước 9: Sao Lưu Về Google Drive

In [ ]:
BACKUP_DIR = os.path.join(DRIVE_DATA_DIR, 'results_backup') if DRIVE_DATA_DIR else '/content/drive/MyDrive/PHEME_results_backup'
os.makedirs(BACKUP_DIR, exist_ok=True)
!cp -r results/* "$BACKUP_DIR/"
print(f'✅ Đã sao lưu kết quả vào: {BACKUP_DIR}')

---
## Bước 10: Kiểm Tra & Download JSON Về Máy

Chạy cell này để xem tóm tắt kết quả và tải về các file JSON cần thiết để cập nhật `Final_Nhóm 6.docx`.

In [ ]:
import json

# File cần tải về (TTT baseline lấy từ máy local)
RESULT_FILES = {
    # UC-TTT
    'UC-TTT (residual_gate)':        'results/ucttt/ucttt_results.json',
    # Ablation — Feature
    'Ablation: counts_only':         'results/ablation/counts_only_results.json',
    'Ablation: without_ratios':      'results/ablation/without_ratios_results.json',
    'Ablation: without_profile':     'results/ablation/without_profile_results.json',
    'Ablation: without_counts':      'results/ablation/without_counts_results.json',
    'Ablation: profile_only':        'results/ablation/profile_only_results.json',
    'Ablation: ratios_only':         'results/ablation/ratios_only_results.json',
    'Ablation: source_only':         'results/ablation/source_only_results.json',
    # Ablation — Architecture
    'Ablation: without_temporal_gru':     'results/ablation/without_temporal_gru_results.json',
    'Ablation: without_tree_transformer': 'results/ablation/without_tree_transformer_results.json',
    # Summary
    'Ablation summary':              'results/ablation/ablation_summary.json',
}

print(f"{'Tên':45s} {'Mean LOEO F1':>14s} {'Pooled F1':>11s} {'Status'}")
print("-" * 85)

to_download = []
for label, fpath in RESULT_FILES.items():
    if os.path.exists(fpath):
        try:
            d = json.load(open(fpath))
            m = d.get('mean_loeo_macro_f1') or d.get('mean_macro_f1', '—')
            p = d.get('pooled_macro_f1', '—')
            m_str = f"{m*100:.2f}%" if isinstance(m, float) else str(m)
            p_str = f"{p*100:.2f}%" if isinstance(p, float) else str(p)
        except Exception:
            m_str = p_str = '—'
        print(f"✅ {label:43s} {m_str:>14s} {p_str:>11s}")
        to_download.append(fpath)
    else:
        print(f"❌ {label:43s} {'—':>14s} {'—':>11s}  (chưa có)")

print(f"\n📦 {len(to_download)}/{len(RESULT_FILES)} file sẵn sàng tải.")

In [ ]:
# Tải tất cả file JSON đã có về máy
from google.colab import files

for fpath in to_download:
    print(f"⬇️  {fpath}")
    files.download(fpath)

print("\n✅ Xong! Copy các file JSON vào đúng thư mục results/ trong repo local,")
print("   sau đó chạy script build để cập nhật Final_Nhóm 6.docx.")